In [9]:
# ── Cell 1 · Install all required packages ────────────────────
!pip install pdfplumber chromadb sentence-transformers \
            anthropic fastapi uvicorn python-multipart \
            aiofiles pypdf -q

import importlib, sys
for pkg in ["pdfplumber", "chromadb", "sentence_transformers", "anthropic"]:
    print(f"✓ {pkg}" if importlib.util.find_spec(pkg) else f"✗ {pkg} MISSING")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 70.1 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 75.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 929.8/929.8 kB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 104.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 81.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━

In [10]:
!pip install groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.4 MB/s eta 0:00:00


In [11]:
!pip install -q chromadb==0.5.5

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 584.3/584.3 kB 15.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 55.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 88.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 279.4/279.4 kB 20.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.29.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requir

In [ ]:
import os
os.kill(os.getpid(), 9)

In [1]:
import os

os.environ["ANONYMIZED_TELEMETRY"] = "False"
os.environ["CHROMA_TELEMETRY_ENABLED"] = "False"
os.environ["OTEL_SDK_DISABLED"] = "true"

In [2]:
# ── Cell 2 · Config — Groq key from Kaggle Secrets ────────────
import os
from pathlib import Path
from kaggle_secrets import UserSecretsClient

PDF_DIR       = Path("/kaggle/input/datasets/shreyash28/indian-law")
CHROMA_DIR    = Path("/kaggle/working/chroma_db")
EMBED_MODEL   = "BAAI/bge-base-en-v1.5"
COLLECTION    = "indian_law"
CHUNK_SIZE    = 350
CHUNK_OVERLAP = 75
TOP_K         = 15
GROQ_MODEL    = "llama-3.3-70b-versatile"# free, fast, good for legal Q&A

# Load from Kaggle Secrets — name must match exactly
secrets   = UserSecretsClient()
GROQ_KEY  = secrets.get_secret("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = GROQ_KEY

CHROMA_DIR.mkdir(parents=True, exist_ok=True)

# Sanity check — should print gsk_...<last 4 chars>
print(f"Key loaded : gsk_...{GROQ_KEY.strip()[-4:]}")
print(f"Key length : {len(GROQ_KEY.strip())} chars (should be ~56)")
print(f"PDFs found : {len(list(PDF_DIR.glob('*.pdf')))}")

Key loaded : gsk_...R9NA
Key length : 56 chars (should be ~56)
PDFs found : 6


In [3]:
import pdfplumber

def extract_pages(pdf_path: Path):
    pages = []

    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            text = page.extract_text()

            if text:
                pages.append({
                    "page": page_num,
                    "text": text.strip()
                })

    return pages

sample = list(PDF_DIR.glob("*.pdf"))[0]
pages = extract_pages(sample)

print(f"Pages extracted: {len(pages)}")
print(pages[0]["text"][:400])

Pages extracted: 41
THE INFORMATION TECHNOLOGY ACT, 2000
–––––––––
ARRANGEMENT OF SECTIONS
–––––––––
CHAPTER I
PRELIMINARY
SECTIONS
1. Short title, extent, commencement and application.
2. Definitions.
CHAPTER II
DIGITAL SIGNATURE AND ELECTRONIC SIGNATURE
3. Authentication of electronic records.
3A. Electronic signature.
CHAPTER III
ELECTRONIC GOVERNANCE
4. Legal recognition of electronic records.
5. Legal recognitio


In [4]:
import re

ACT_MAP = {
    "constitution_of_india.pdf":
        "Constitution of India",

    "the_bharatiya_nagarik_suraksha_sanhita_2023.pdf":
        "Bharatiya Nagarik Suraksha Sanhita, 2023"
}

def get_act_name(filename):

    return ACT_MAP.get(
        filename.lower(),
        Path(filename).stem.replace("_", " ").title()
    )

def extract_metadata(text):

    meta = {}

    part = re.search(
        r"PART\s+([IVXLC]+)",
        text,
        re.I
    )

    if part:
        meta["part"] = f"Part {part.group(1)}"

    chapter = re.search(
        r"CHAPTER\s+([IVXLC]+)",
        text,
        re.I
    )

    if chapter:
        meta["chapter"] = f"Chapter {chapter.group(1)}"

    article = re.search(
        r"Article\s+(\d+[A-Z]?)",
        text,
        re.I
    )

    if article:
        meta["article"] = article.group(1)

    section = re.search(
        r"Section\s+(\d+[A-Z]?)",
        text,
        re.I
    )

    if section:
        meta["section"] = section.group(1)

    return meta

def chunk_page(page_text,
               page_num,
               source):

    words = page_text.split()

    chunks = []

    start = 0

    running_meta = {}

    while start < len(words):

        end = min(
            start + CHUNK_SIZE,
            len(words)
        )

        chunk = " ".join(words[start:end])

        current = extract_metadata(chunk)

        running_meta.update(current)

        chunks.append({
            "text": chunk,
            "source": source,
            "page": page_num,
            **running_meta
        })

        start += CHUNK_SIZE - CHUNK_OVERLAP

    return chunks

In [5]:
import chromadb
print(chromadb.__version__)

0.5.5


In [6]:
# ── Cell 5 · Load embedding model — auto GPU/CPU ──────────────
import torch
from sentence_transformers import SentenceTransformer

# Auto-select: uses GPU if available (T4), falls back to CPU
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"CUDA available : {torch.cuda.is_available()}")
print(f"Using device   : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    print(f"GPU count      : {torch.cuda.device_count()}")

embed_model = SentenceTransformer(EMBED_MODEL, device=DEVICE)
print(f"Model loaded   : {EMBED_MODEL}")
print(f"Embedding dim  : {embed_model.get_embedding_dimension()}")

# Smoke test
test_vec = embed_model.encode(
    "What is Article 21 of the Indian Constitution?",
    device=DEVICE
)
print(f"Test vector    : shape={test_vec.shape}, dtype={test_vec.dtype}")

CUDA available : True
Using device   : cuda
GPU            : Tesla T4
GPU count      : 2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded   : BAAI/bge-base-en-v1.5
Embedding dim  : 768
Test vector    : shape=(768,), dtype=float32


In [7]:
# ── Cell 6 · ChromaDB — works as-is on T4 ─────────────────────
# ChromaDB runs its embedding function on CPU internally anyway.
# The device="cpu" argument here is correct and intentional —
# don't change it. GPU is used by Cell 5's embed_model only.
import chromadb
from chromadb.utils import embedding_functions

ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name=EMBED_MODEL,
    device="cpu"   # ChromaDB internal — leave as cpu
)

chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = chroma_client.get_or_create_collection(
    name=COLLECTION,
    embedding_function=ef,
    metadata={"hnsw:space": "cosine"}
)
print(f"Collection   : {collection.name}")
print(f"Chunks stored: {collection.count()}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Collection   : indian_law
Chunks stored: 0


In [8]:
import hashlib

def pdf_md5(path):

    h = hashlib.md5()

    with open(path, "rb") as f:

        for block in iter(
            lambda: f.read(65536),
            b""
        ):
            h.update(block)

    return h.hexdigest()

def ingest_pdf(pdf_path):

    md5 = pdf_md5(pdf_path)

    existing = collection.get(
        where={"pdf_hash": md5},
        limit=1
    )

    if existing["ids"]:
        print(f"⏭ skip {pdf_path.name}")
        return

    pages = extract_pages(pdf_path)

    act_name = get_act_name(
        pdf_path.name
    )

    docs = []
    metas = []
    ids = []

    idx = 0

    for page in pages:

        chunks = chunk_page(
            page["text"],
            page["page"],
            pdf_path.name
        )

        for c in chunks:

            ids.append(
                f"{md5}_{idx}"
            )

            docs.append(
                c["text"]
            )

            meta = {
                "source": pdf_path.name,
                "act_name": act_name,
                "page": int(c.get("page", 0)),
                "pdf_hash": md5
            }
            
            if c.get("part"):
                meta["part"] = str(c["part"])
            
            if c.get("chapter"):
                meta["chapter"] = str(c["chapter"])
            
            if c.get("article"):
                meta["article"] = str(c["article"])
            
            if c.get("section"):
                meta["section"] = str(c["section"])
            
            metas.append(meta)

            idx += 1

    for i in range(
        0,
        len(ids),
        500
    ):

        collection.add(
            ids=ids[i:i+500],
            documents=docs[i:i+500],
            metadatas=metas[i:i+500]
        )

    print(
        f"✅ {pdf_path.name} "
        f"({len(ids)} chunks)"
    )

print("Ingesting PDFs...")

for pdf in sorted(
    PDF_DIR.glob("*.pdf")
):
    ingest_pdf(pdf)

print(
    f"\nTotal chunks: "
    f"{collection.count()}"
)

Ingesting PDFs...


Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


✅ A2000-21 (1).pdf (97 chunks)
✅ a202345.pdf (307 chunks)
✅ c9fe9c9b6840524844316f74bb1c556c.pdf (712 chunks)
✅ ca7ce5c746fa7480804bbdeb6cb704f0.pdf (719 chunks)
✅ constitution_of_india.pdf (502 chunks)
✅ the_bharatiya_nagarik_suraksha_sanhita_2023.pdf (629 chunks)

Total chunks: 2966


In [9]:
# ── Cell 8 · Retrieve top-K chunks for any query ──────────────
def retrieve(
    query: str,
    top_k: int = 15
):

    if collection.count() == 0:
        return []

    results = collection.query(
        query_texts=[query],
        n_results=min(
            top_k,
            collection.count()
        ),
        include=[
            "documents",
            "metadatas",
            "distances"
        ]
    )

    hits = []

    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):

        hits.append({
            "text": doc,
            "source": meta.get("source"),
            "act_name": meta.get("act_name"),
            "part": meta.get("part"),
            "chapter": meta.get("chapter"),
            "article": meta.get("article"),
            "section": meta.get("section"),
            "page": meta.get("page"),
            "score": round(
                1 - dist,
                4
            )
        })

    return hits
# Test it
query = "What are the fundamental rights under the Indian Constitution?"
hits  = retrieve(query)
print(f"Query: {query}\n")
for i, h in enumerate(hits, 1):
    print(f"[{i}] score={h['score']} | {h['source']}")
    print(h["text"][:200], "…\n")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Query: What are the fundamental rights under the Indian Constitution?

[1] score=0.78 | constitution_of_india.pdf
THE CONSTITUTION OF INDIA 17 (Part III.—Fundamental Rights.—Arts. 31A—31C.) (ii) any land held under ryotwari settlement; (iii) any land held or let for purposes of agriculture or for purposes ancilla …

[2] score=0.7745 | ca7ce5c746fa7480804bbdeb6cb704f0.pdf
PART III FUNDAMENTAL RIGHTS General 12. Definition.—In this Part, unless the context otherwise requires, “the State” includes the Government and Parliament of India and the Government and the Legislat …

[3] score=0.7703 | c9fe9c9b6840524844316f74bb1c556c.pdf
PART III FUNDAMENTAL RIGHTS General 12. Definition.—In this Part, unless the context otherwise requires, “the State” includes the Government and Parliament of India and the Government and the Legislat …

[4] score=0.7679 | constitution_of_india.pdf
THE CONSTITUTION OF INDIA 7 (Part III.—Fundamental Rights.—Arts. 15-16.) 15. (1) The State shall not discriminate aga

In [10]:
!pip install sentence-transformers -q

In [11]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    device=DEVICE
)

def rerank(query, hits, top_n=5):
    if not hits:
        return hits

    pairs = [[query, h["text"]] for h in hits]

    scores = reranker.predict(pairs)

    for h, s in zip(hits, scores):
        h["rerank_score"] = float(s)

    hits.sort(
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    return hits[:top_n]

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [12]:
# ── Cell 9 · Answer generation via Groq (replaces Anthropic) ──
from groq import Groq

SYSTEM = """
You are an expert Indian Law assistant.

STRICT RULES:

1. Use ONLY the retrieved legal context.
2. Do NOT use prior legal knowledge.
3. Do NOT infer missing articles, sections, or provisions.
4. If the answer is not completely supported by the retrieved context,
   explicitly state:
   "The retrieved documents do not contain sufficient information."
5. Cite the source document whenever making a legal statement.
6. Never invent citations.

End every answer with:

This is informational only, not legal advice.
"""
def build_context(hits):

    if not hits:
        return "No relevant documents found."

    blocks = []

    for i, h in enumerate(hits, start=1):

        header = (
            f"[Source {i}] "
            f"Act={h.get('act_name')} | "
            f"Part={h.get('part')} | "
            f"Article={h.get('article')} | "
            f"Section={h.get('section')} | "
            f"Page={h.get('page')}"
        )

        blocks.append(
            header + "\n" + h["text"]
        )

    return "\n\n---\n\n".join(blocks)
def ask(question: str) -> str:
    hits = retrieve(question, top_k=20)
    hits = rerank(question, hits, top_n=5)
    context = build_context(hits)
    client  = Groq(api_key=GROQ_KEY.strip())   # .strip() removes accidental whitespace
    response = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user",   "content": f"Question: {question}\n\nContext:\n{context}"}
        ],
        temperature=0.2,    # low = more factual, good for legal
        max_tokens=1500
    )
    return response.choices[0].message.content

# Test
answer = ask("What are the fundamental rights under the Indian Constitution?")
print(answer)

The fundamental rights under the Indian Constitution are listed in Part III of the Constitution, which includes:

1. Right to Equality (Articles 14-18) - This includes equality before the law, prohibition of discrimination on grounds of religion, race, caste, sex, or place of birth, and equality of opportunity in matters of public employment. (Source 3, Article 15; Source 4, Article 14)
2. Right to Freedom (Article 19) - This includes the right to freedom of speech and expression, assembly, association, movement, residence, and profession. (Source 1, Article 19)
3. Protection of certain rights regarding freedom of speech, etc. (Article 19) - This includes reasonable restrictions on the exercise of the right to freedom of speech and expression in the interests of the sovereignty and integrity of India, security of the State, friendly relations with foreign States, public order, decency, or morality. (Source 1, Article 19)
4. Protection in respect of conviction for offences (Article 20) 

In [13]:
# ── Cell 10 · Add a new PDF at any time without full re-index ─
def add_new_pdf(pdf_path: str):
    path = Path(pdf_path)
    assert path.exists(), f"File not found: {path}"
    assert path.suffix.lower() == ".pdf", "Only PDF files supported"
    ingest_pdf(path)
    print(f"\nKnowledge base now has {collection.count()} chunks")

# Usage:
# add_new_pdf("/path/to/new_act.pdf")

# ── Stats helper ───────────────────────────────────────────────
def kb_stats():
    n = collection.count()
    if n == 0:
        print("Knowledge base is empty."); return
    meta   = collection.get(limit=n, include=["metadatas"])
    files  = sorted({m["source"] for m in meta["metadatas"]})
    print(f"Documents : {len(files)}")
    print(f"Chunks    : {n}")
    print(f"Files     :")
    for f in files: print(f"  · {f}")

kb_stats()

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


Documents : 6
Chunks    : 2966
Files     :
  · A2000-21 (1).pdf
  · a202345.pdf
  · c9fe9c9b6840524844316f74bb1c556c.pdf
  · ca7ce5c746fa7480804bbdeb6cb704f0.pdf
  · constitution_of_india.pdf
  · the_bharatiya_nagarik_suraksha_sanhita_2023.pdf
